# Ejercicio 2 — Transformaciones de Variables Normales

## Máster Executive en Finanzas Cuantitativas 2026 — AFI Global Education
### Fundamentos Matemáticos: Probabilidad y Simulación

---

### Enunciado

Sea $X\sim\mathcal N(0,1)$. Consideramos las variables transformadas $Y=g(X)$ y $Z=h(X)$, con

$$g(x)=\begin{cases}-x & x<0\\[2pt]\alpha x & x\ge 0\end{cases}\qquad\qquad h(x)=\begin{cases}-x & x<0\\[2pt]\sqrt{x} & x\ge 0\end{cases}$$

### Hoja de ruta de la resolución

| Apartado | Qué se pide | Herramienta |
|:--:|---|---|
| **2.1** | Densidad de $Y$ por simulación para $\alpha\in\{-1,-\tfrac12,0,\tfrac12,1\}$; ¿cuándo es $Y$ continua? | Simulación + histograma |
| **2.2** | Derivar **analíticamente** $f_Y(y)$ para $\alpha>0$ | Método CDF → derivar |
| **2.3** | **Demostrar** la fórmula de $f_Z(z)$ y comprobarla | Método CDF → derivar + test KS |

**Idea global.** Cuando aplicamos una función $g$ a una variable aleatoria $X$, la variable resultante $Y=g(X)$ tiene una densidad *distinta* a la de $X$. La técnica maestra para hallarla es el **método de la función de distribución**: se escribe $F_Y(y)=P(Y\le y)$ traduciéndolo a un suceso sobre $X$ (cuya distribución sí conocemos) y luego se **deriva** respecto a $y$ para obtener $f_Y=F_Y'$. Las transformaciones son *a tramos*, así que habrá que sumar la contribución de cada tramo.

---
## Configuración del entorno

Una única celda con *imports*, estilo gráfico, **semilla fija** y constantes. La normal estándar $X$ se simula una sola vez y se reutiliza en todo el notebook.

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, integrate
import os

# ── Estilo gráfico coherente ────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

# ── Semilla fija (reproducibilidad) ─────────────────────────────────────────
SEED = 42
rng  = np.random.default_rng(SEED)

# ── Constantes del problema ─────────────────────────────────────────────────
N_SIM  = 100_000                       # nº de muestras para simular densidades
ALPHAS = [-1, -0.5, 0, 0.5, 1]          # valores de α pedidos en el enunciado

# ── Carpeta de figuras ──────────────────────────────────────────────────────
os.makedirs("resultados", exist_ok=True)

# ── Base común: una sola muestra de X ~ N(0,1), reutilizada en todo ──────────
X = rng.standard_normal(N_SIM)
print(f"Entorno listo · SEED={SEED} · N_SIM={N_SIM:,}")
print(f"X ~ N(0,1):  media={X.mean():.4f}  std={X.std():.4f}   (esperado 0 y 1)")

---
## Apartado 2.1 — Densidad de $Y=g(X)$ por simulación

### Lectura de la transformación

$$g(x)=\begin{cases}-x & x<0\\\alpha x & x\ge 0\end{cases}$$

La transformación trata cada mitad de la recta de forma distinta:

- **Rama izquierda** ($x<0$): $g(x)=-x>0$. Es una **reflexión** que manda la cola negativa de $X$ al semieje positivo.
- **Rama derecha** ($x\ge 0$): $g(x)=\alpha x$. Es un **reescalado** por el factor $\alpha$.

El signo y el valor de $\alpha$ deciden qué aspecto tiene $Y$. La siguiente tabla anticipa el análisis (que las simulaciones confirmarán):

| $\alpha$ | Qué hace la rama derecha | Soporte de $Y$ | ¿$Y$ continua? |
|:--:|---|:--:|:--:|
| $-1$ | $g(x)=-x$ en **todo** $\mathbb R$ → $Y=-X$ | $\mathbb R$ | **Sí** ($Y\sim\mathcal N(0,1)$) |
| $-0.5$ | manda $x\ge0$ a valores **negativos** | $\mathbb R$ | **Sí** (asimétrica) |
| $0$ | $g(x)=0$ para **todo** $x\ge0$ | $\{0\}\cup(0,\infty)$ | **No** (masa puntual en 0) |
| $0.5$ | ambas ramas en $(0,\infty)$ | $(0,\infty)$ | **Sí** (asimétrica) |
| $1$ | $g(x)=|x|$ → $Y=|X|$ | $[0,\infty)$ | **Sí** (*half-normal*) |

**Conclusión clave:** $Y$ es una variable aleatoria **continua si y sólo si $\alpha\neq 0$**. Los dos casos especiales del enunciado se analizan al final del apartado.

In [ ]:
# ── Transformación g y simulación de Y para cada α ──────────────────────────
def transform_g(x, alpha):
    """Y = g(X): refleja la rama negativa (-x) y reescala la positiva (αx)."""
    return np.where(x < 0, -x, alpha * x)

# Aplicamos g a la MISMA muestra X para cada α (comparabilidad)
Y_sim = {a: transform_g(X, a) for a in ALPHAS}

# Diagnóstico numérico: media, soporte y masa puntual en 0
print(f"{'α':>5} | {'media':>8} {'min':>8} {'max':>8} | {'P(Y=0)':>8}  observación")
print("-" * 64)
for a in ALPHAS:
    Y = Y_sim[a]
    p0 = np.mean(Y == 0)
    nota = "masa puntual → mixta" if p0 > 0.01 else ("= N(0,1)" if a == -1 else "continua")
    print(f"{a:>5} | {Y.mean():>8.4f} {Y.min():>8.3f} {Y.max():>8.3f} | {p0:>8.4f}  {nota}")

In [ ]:
# ── Figura 1: densidad simulada de Y para cada α (con N(0,1) de referencia) ──
fig, axes = plt.subplots(1, 5, figsize=(18, 3.8))

for ax, a in zip(axes, ALPHAS):
    Y = Y_sim[a]
    lo, hi = np.percentile(Y, [0.3, 99.7])
    ax.hist(Y, bins=70, range=(lo, hi), density=True, alpha=0.6,
            color="steelblue", label="simulación")
    # Referencia: densidad N(0,1)
    xr = np.linspace(lo, hi, 300)
    ax.plot(xr, stats.norm.pdf(xr), "r--", lw=1.5, label=r"$N(0,1)$")
    # Si α=0, marcar la masa puntual en 0 con una flecha
    if a == 0:
        ax.annotate("masa\npuntual\n$P(Y{=}0){=}\\frac{1}{2}$", xy=(0, ax.get_ylim()[1]*0.9),
                    xytext=(0.6, ax.get_ylim()[1]*0.7), fontsize=8,
                    arrowprops=dict(arrowstyle="->", color="darkred"))
    ax.set_title(f"$\\alpha={a}$", fontsize=12)
    ax.set_xlabel("y")
    if ax is axes[0]:
        ax.set_ylabel("densidad")
    ax.legend(fontsize=8)

fig.suptitle(r"Densidad simulada de $Y=g(X)$ según $\alpha$  —  $X\sim N(0,1)$", fontsize=13)
fig.tight_layout()
fig.savefig("resultados/grafico_04_fY_por_alpha.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada en resultados/grafico_04_fY_por_alpha.png")

### Análisis de continuidad y los casos especiales $\alpha=0$ y $\alpha=1$

**¿Para qué $\alpha$ es $Y$ continua?** Para que $Y$ sea continua, su recorrido no puede concentrar probabilidad en ningún punto. La rama derecha $g(x)=\alpha x$ con $x\ge0$:

- Si $\alpha\neq0$, mapea $[0,\infty)$ de forma *biyectiva* sobre un intervalo (no colapsa nada) → $Y$ continua.
- Si $\alpha=0$, manda **todo** el semieje $x\ge0$ al **único punto** $\{0\}$ → concentra ahí toda la probabilidad de $\{X\ge0\}$.

Por tanto, **$Y$ es continua $\iff \alpha\neq0$**.

**Caso $\alpha=0$ (variable mixta).** Como $X\sim\mathcal N(0,1)$ es simétrica, $P(X\ge0)=\tfrac12$. Toda esa mitad se aplasta en $Y=0$, generando una **masa puntual**:

$$P(Y=0)=P(X\ge0)=\tfrac12.$$

La simulación lo confirma ($P(Y{=}0)\approx 0.496$). $Y$ no es ni puramente discreta ni puramente continua: es **mixta** (un átomo de probabilidad $\tfrac12$ en 0, más una parte continua en $(0,\infty)$ que proviene de la rama reflejada $-x$).

**Caso $\alpha=1$ (half-normal).** Aquí $g(x)=-x$ para $x<0$ y $g(x)=x$ para $x\ge0$, es decir $g(x)=|x|$, luego $Y=|X|$. El valor absoluto de una normal estándar sigue la distribución **semi-normal** (*half-normal*), con densidad

$$f_Y(y)=2\,\phi(y)=\sqrt{\tfrac{2}{\pi}}\,e^{-y^2/2},\quad y\ge0,$$

(el factor 2 recoge que las dos colas de $X$, en $-y$ y en $+y$, se suman sobre el mismo $y>0$). Su media teórica es $\sqrt{2/\pi}\approx0.798$, que coincide con la simulación. Este caso $\alpha=1$ es además la **frontera** entre los dos comportamientos del apartado siguiente: lo recuperaremos como caso particular de la fórmula general de $f_Y$.

---
## Apartado 2.2 — Derivación analítica de $f_Y(y)$ para $\alpha>0$

Seguimos la sugerencia del enunciado: **escribir $P(Y<y)$ y derivar sobre $y$**.

### Paso 1 — El soporte: $Y$ sólo toma valores positivos

Con $\alpha>0$, las dos ramas producen valores $\ge0$: la izquierda da $-x>0$ (porque $x<0$) y la derecha da $\alpha x\ge0$. Luego $Y\in[0,\infty)$ y basta estudiar $y>0$.

### Paso 2 — Traducir $\{Y\le y\}$ a un suceso sobre $X$

El suceso $\{Y\le y\}$ se descompone según la rama por la que pasó $X$:

$$\{Y\le y\}=\underbrace{\{X<0,\;-X\le y\}}_{\text{rama izquierda}}\;\cup\;\underbrace{\{X\ge0,\;\alpha X\le y\}}_{\text{rama derecha}}$$

Despejando $X$ en cada parte (son sucesos disjuntos, así que las probabilidades se suman):

$$F_Y(y)=P(Y\le y)=\underbrace{P(-y\le X<0)}_{\text{de }-X\le y}\;+\;\underbrace{P\!\left(0\le X\le \tfrac{y}{\alpha}\right)}_{\text{de }\alpha X\le y}$$

### Paso 3 — Escribir con la CDF normal $\Phi$

$$F_Y(y)=\bigl[\Phi(0)-\Phi(-y)\bigr]+\bigl[\Phi(\tfrac{y}{\alpha})-\Phi(0)\bigr]=\Phi\!\left(\frac{y}{\alpha}\right)-\Phi(-y)$$

(los dos $\Phi(0)=\tfrac12$ se cancelan).

### Paso 4 — Derivar para obtener la densidad

Usando $\frac{d}{dy}\Phi(u(y))=\phi(u(y))\,u'(y)$ y que $\phi$ es par ($\phi(-y)=\phi(y)$):

$$f_Y(y)=\frac{d}{dy}\Phi\!\left(\frac{y}{\alpha}\right)-\frac{d}{dy}\Phi(-y)=\frac{1}{\alpha}\phi\!\left(\frac{y}{\alpha}\right)-\phi(-y)\cdot(-1)$$

$$\boxed{\;f_Y(y)=\frac{1}{\alpha}\,\phi\!\left(\frac{y}{\alpha}\right)+\phi(y),\qquad y>0,\;\alpha>0.\;}$$

### Interpretación y verificaciones

La densidad es **suma de dos campanas**: $\phi(y)$ es la contribución de la cola negativa reflejada, y $\tfrac1\alpha\phi(y/\alpha)$ es la contribución de la cola positiva reescalada (el factor $1/\alpha$ es el jacobiano del cambio $y=\alpha x$).

- **Normalización:** $\int_0^\infty f_Y=\tfrac12+\tfrac12=1$ (cada campana aporta media unidad sobre $y>0$).
- **Caso $\alpha=1$:** $f_Y(y)=\phi(y)+\phi(y)=2\phi(y)$ → recuperamos la **half-normal** del apartado 2.1. ✓

In [ ]:
# ── Densidad analítica de Y para α>0 y sus verificaciones ───────────────────
def f_Y(y, alpha):
    """f_Y(y) = (1/α)·φ(y/α) + φ(y),  válida para y>0, α>0."""
    assert alpha > 0, "Esta fórmula sólo vale para α>0"
    return (1/alpha) * stats.norm.pdf(y/alpha) + stats.norm.pdf(y)

# Verificación 1: la densidad integra 1 para varios α
print("Verificación de normalización  ∫₀^∞ f_Y(y) dy = 1:")
for a in [0.5, 1, 2, 3]:
    val, _ = integrate.quad(lambda y: f_Y(y, a), 0, np.inf)
    print(f"  α={a}:  {val:.10f}")
    assert abs(val - 1) < 1e-8

# Verificación 2: caso α=1 debe coincidir con la half-normal 2·φ(y)
y_test = np.array([0.3, 1.0, 2.0])
print(f"\nCaso α=1 vs half-normal 2·φ(y):")
print(f"  f_Y(y,1)  = {f_Y(y_test, 1)}")
print(f"  2·φ(y)    = {2*stats.norm.pdf(y_test)}")
assert np.allclose(f_Y(y_test, 1), 2*stats.norm.pdf(y_test))
print("✓ Coinciden exactamente")

In [ ]:
# ── Figura 2: f_Y analítica superpuesta al histograma simulado (α>0) ────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

for ax, a in zip(axes, [0.5, 1.0]):
    Y = Y_sim[a]
    Y_pos = Y[Y > 0]                               # parte positiva (toda, si α>0)
    ax.hist(Y_pos, bins=80, density=True, alpha=0.55, color="steelblue",
            label=f"simulación (N={N_SIM:,})")
    yy = np.linspace(1e-3, 4, 400)
    ax.plot(yy, f_Y(yy, a), "r-", lw=2.5,
            label=r"$f_Y(y)=\frac{1}{\alpha}\phi(\frac{y}{\alpha})+\phi(y)$")
    titulo = f"$\\alpha={a}$" + ("  (= half-normal $2\\phi$)" if a == 1 else "")
    ax.set_title(titulo, fontsize=12)
    ax.set_xlabel("y"); ax.set_ylabel("densidad")
    ax.legend(fontsize=9)

fig.suptitle(r"Apartado 2.2 — densidad analítica de $Y$ vs simulación ($\alpha>0$)", fontsize=13)
fig.tight_layout()
fig.savefig("resultados/grafico_05_fY_analitica_vs_sim.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada en resultados/grafico_05_fY_analitica_vs_sim.png")

---
## Apartado 2.3 — Densidad de $Z=h(X)$: demostración y comprobación

Hay que **demostrar** que $\displaystyle f_Z(z)=\frac{1}{\sqrt{2\pi}}\bigl(2z\,e^{-z^4/2}+e^{-z^2/2}\bigr)$ y comprobarlo por simulación. Usamos el mismo método (CDF → derivar).

### Paso 1 — Soporte

$$h(x)=\begin{cases}-x & x<0\\\sqrt{x} & x\ge0\end{cases}$$

La rama izquierda da $-x>0$ y la derecha $\sqrt{x}\ge0$, luego $Z\in[0,\infty)$; estudiamos $z>0$.

### Paso 2 — Traducir $\{Z\le z\}$ a un suceso sobre $X$

$$\{Z\le z\}=\{X<0,\;-X\le z\}\cup\{X\ge0,\;\sqrt X\le z\}$$

La condición $\sqrt X\le z$ con $X\ge0$ equivale a $X\le z^2$. Despejando:

$$F_Z(z)=P(-z\le X<0)+P(0\le X\le z^2)$$

### Paso 3 — Escribir con $\Phi$

$$F_Z(z)=\bigl[\Phi(0)-\Phi(-z)\bigr]+\bigl[\Phi(z^2)-\Phi(0)\bigr]=\Phi(z^2)-\Phi(-z)$$

### Paso 4 — Derivar (regla de la cadena)

El primer término lleva $z^2$ dentro, cuya derivada es $2z$; el segundo lleva $-z$, derivada $-1$:

$$f_Z(z)=\frac{d}{dz}\Phi(z^2)-\frac{d}{dz}\Phi(-z)=\phi(z^2)\cdot 2z-\phi(-z)\cdot(-1)=2z\,\phi(z^2)+\phi(z)$$

### Paso 5 — Sustituir la densidad normal $\phi$

Como $\phi(t)=\frac{1}{\sqrt{2\pi}}e^{-t^2/2}$, tenemos $\phi(z^2)=\frac{1}{\sqrt{2\pi}}e^{-(z^2)^2/2}=\frac{1}{\sqrt{2\pi}}e^{-z^4/2}$ y $\phi(z)=\frac{1}{\sqrt{2\pi}}e^{-z^2/2}$. Sustituyendo:

$$\boxed{\;f_Z(z)=\frac{1}{\sqrt{2\pi}}\bigl(2z\,e^{-z^4/2}+e^{-z^2/2}\bigr),\qquad z>0.\;}\qquad\blacksquare$$

que es exactamente la fórmula del enunciado.

### Verificación de normalización (analítica)

Que $f_Z$ integre 1 confirma la demostración. El primer sumando, con el cambio $u=z^2$ ($du=2z\,dz$):

$$\int_0^\infty\frac{2z}{\sqrt{2\pi}}e^{-z^4/2}dz=\frac{1}{\sqrt{2\pi}}\int_0^\infty e^{-u^2/2}du=\frac{1}{\sqrt{2\pi}}\cdot\frac{\sqrt{2\pi}}{2}=\frac12,$$

y el segundo es $\int_0^\infty\phi(z)dz=\tfrac12$. Total $=1$. ✓

In [ ]:
# ── Transformación h, densidad y CDF analíticas de Z ────────────────────────
def transform_h(x):
    """Z = h(X): -x si x<0, sqrt(x) si x>=0."""
    return np.where(x < 0, -x, np.sqrt(np.maximum(x, 0)))

def f_Z(z):
    """Densidad demostrada: (1/√2π)(2z·e^{-z⁴/2} + e^{-z²/2})."""
    return (1/np.sqrt(2*np.pi)) * (2*z*np.exp(-z**4/2) + np.exp(-z**2/2))

def F_Z(z):
    """CDF: Φ(z²) − Φ(−z).  La usamos para el test de Kolmogórov–Smirnov."""
    return stats.norm.cdf(z**2) - stats.norm.cdf(-z)

# Simular Z aplicando h a la muestra base X
Z = transform_h(X)

# Verificación A — normalización numérica
area_Z, _ = integrate.quad(f_Z, 0, np.inf)
print(f"∫₀^∞ f_Z(z) dz = {area_Z:.10f}   (debe ser 1)")
assert abs(area_Z - 1) < 1e-7

# Verificación B — la derivada de F_Z coincide con f_Z (confirma el paso 4)
z0, eps = 1.3, 1e-6
deriv_num = (F_Z(z0+eps) - F_Z(z0-eps)) / (2*eps)
print(f"\nd F_Z/dz en z={z0}:  numérica={deriv_num:.6f}  vs  f_Z={f_Z(z0):.6f}")
assert abs(deriv_num - f_Z(z0)) < 1e-4
print("✓ La derivada de la CDF reproduce f_Z → demostración consistente")

In [ ]:
# ── Comprobación por simulación: test de Kolmogórov–Smirnov ─────────────────
# El KS compara la CDF empírica de las muestras Z con la CDF teórica F_Z.
ks_stat, ks_pval = stats.kstest(Z, F_Z)
print(f"Test de Kolmogórov–Smirnov  (simulación vs teoría):")
print(f"  estadístico D = {ks_stat:.5f}")
print(f"  p-valor       = {ks_pval:.4f}")
print(f"  → {'NO se rechaza' if ks_pval > 0.05 else 'se rechaza'} "
      f"que Z siga la densidad demostrada (nivel 5%)")

In [ ]:
# ── Figura 3: f_Z simulada vs analítica + error histograma–teoría ───────────
fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 4.2))

# (izq.) histograma de Z con la curva teórica encima
axL.hist(Z, bins=100, range=(0, 4), density=True, alpha=0.55, color="steelblue",
         label=f"simulación (N={N_SIM:,})")
zz = np.linspace(1e-3, 4, 500)
axL.plot(zz, f_Z(zz), "r-", lw=2.5,
         label=r"$f_Z(z)=\frac{1}{\sqrt{2\pi}}(2ze^{-z^4/2}+e^{-z^2/2})$")
axL.set_xlabel("z"); axL.set_ylabel("densidad")
axL.set_title(f"$f_Z$: simulación vs teoría   (KS p={ks_pval:.2f})")
axL.legend(fontsize=9)

# (der.) diferencia absoluta |empírico − teórico| por bin
counts, edges = np.histogram(Z, bins=100, range=(0, 4), density=True)
mids = (edges[:-1] + edges[1:]) / 2
axR.bar(mids, np.abs(counts - f_Z(mids)), width=edges[1]-edges[0],
        color="orange", alpha=0.8)
axR.set_xlabel("z"); axR.set_ylabel("|empírico − teórico|")
axR.set_title(f"Error de ajuste (máx ≈ {np.abs(counts-f_Z(mids)).max():.3f})")

fig.suptitle(r"Apartado 2.3 — verificación de $f_Z(z)$ por simulación", fontsize=13)
fig.tight_layout()
fig.savefig("resultados/grafico_06_fZ_verificacion.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada en resultados/grafico_06_fZ_verificacion.png")

---
## Conclusiones

Tabla-resumen generada **desde las variables del notebook** (sin cifras escritas a mano).

In [ ]:
from IPython.display import display, HTML

filas = [
    ("Apartado", "Resultado", "Valor / observación"),
    ("2.1", "$Y$ es continua",            "si y sólo si α ≠ 0"),
    ("2.1", "caso α = 0",                  f"mixta: P(Y=0)={np.mean(Y_sim[0]==0):.3f} ≈ 1/2"),
    ("2.1", "caso α = 1",                  f"half-normal, media≈{Y_sim[1].mean():.3f} (√(2/π)={np.sqrt(2/np.pi):.3f})"),
    ("2.1", "caso α = −1",                 f"Y=−X ~ N(0,1), media≈{Y_sim[-1].mean():.3f}"),
    ("2.2", "$f_Y(y)$ para α>0",           "(1/α)·φ(y/α) + φ(y),  y>0"),
    ("2.2", "normalización $f_Y$",         "∫ = 1 ✓ (verificado para α=0.5,1,2,3)"),
    ("2.3", "$f_Z(z)$ demostrada",         "(1/√2π)(2z·e^{−z⁴/2} + e^{−z²/2}),  z>0"),
    ("2.3", "normalización $f_Z$",         f"∫ = {area_Z:.6f} ≈ 1 ✓"),
    ("2.3", "test KS (sim. vs teoría)",    f"D={ks_stat:.4f}, p={ks_pval:.3f} → no se rechaza"),
]

html = "<table style='border-collapse:collapse;font-size:13px'>"
for i, (a, b, c) in enumerate(filas):
    if i == 0:
        html += "<tr style='background:#264653;color:white;font-weight:bold'>"
        html += f"<td style='padding:6px 12px'>{a}</td><td style='padding:6px 12px'>{b}</td><td style='padding:6px 12px'>{c}</td></tr>"
    else:
        bg = "#f4f1de" if i % 2 else "white"
        html += f"<tr style='background:{bg}'><td style='padding:6px 12px'>{a}</td><td style='padding:6px 12px'>{b}</td><td style='padding:6px 12px'>{c}</td></tr>"
html += "</table>"
display(HTML(html))

### Discusión final

**Apartado 2.1 — Continuidad.** La simulación para los cinco valores de $\alpha$ muestra que $Y$ cambia de forma radicalmente con el parámetro. La conclusión es nítida: $Y$ es **continua si y sólo si $\alpha\neq0$**. El caso $\alpha=0$ rompe la continuidad porque la rama $x\ge0$ colapsa media unidad de probabilidad en el punto $\{0\}$, dando una variable **mixta** con $P(Y{=}0)=\tfrac12$. El caso $\alpha=1$ corresponde a $Y=|X|$, la distribución **half-normal**.

**Apartado 2.2 — Fórmula de $f_Y$.** El método CDF→derivar conduce a $f_Y(y)=\tfrac1\alpha\phi(y/\alpha)+\phi(y)$ para $y>0$, una **suma de dos campanas** (cola reflejada + cola reescalada). La fórmula integra 1 y, evaluada en $\alpha=1$, reproduce la half-normal $2\phi(y)$, enlazando de forma coherente con el apartado anterior. La superposición de la curva analítica sobre el histograma confirma el resultado.

**Apartado 2.3 — Demostración de $f_Z$.** El mismo método demuestra la fórmula del enunciado: la clave es que la rama $\sqrt{x}$ introduce $z^2$ dentro de $\Phi$, y al derivar aparece el factor $2z$ que da el término $2z\,e^{-z^4/2}$. La demostración se valida por tres vías independientes: normalización analítica ($\int f_Z=1$), coincidencia de la derivada numérica de $F_Z$ con $f_Z$, y un **test de Kolmogórov–Smirnov** que no rechaza la densidad teórica ($p>0.05$). Teoría, cálculo y simulación quedan en total acuerdo.